# Matrix completion via recommendation system example

This example demonstrates the use of matrix completion techniques on a recommendation system.  The recommendation system uses data from the [360K Last.fm dataset](http://ocelma.net/MusicRecommendationDataset/lastfm-360K.html).

In [ ]:
#!pip install -U implicit h5py

In [1]:
# retrieving last.fm dataset
from implicit.datasets.lastfm import get_lastfm
import numpy as np
import pandas as pd
from scipy import sparse
import os
from pathlib import Path

/Users/evanbarnett/Desktop/Northwestern/Classes/de300/data_eng300SP_barnett/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Downloading and saving the Last.fm dataset

In [2]:
filepath = r'datasets/'
Path(filepath).mkdir(exist_ok=True)

if not os.path.exists(filepath + r'artist_user_plays.npz'):
    # save our dataset in sparse format
    artists, users, artist_user_plays = get_lastfm()

    sparse.save_npz(filepath + r'artist_user_plays.npz', artist_user_plays)
    np.save(filepath + 'artists.npy', artists)
    np.save(filepath + 'users.npy', users)
else:
    # load our dataset into original format
    artist_user_plays = sparse.load_npz(filepath + r'artist_user_plays.npz')
    artists = np.load(filepath + 'artists.npy', allow_pickle=True)
    users = np.load(filepath + 'users.npy', allow_pickle=True)

0.00B [00:00, ?B/s]

184MB [00:03, 53.3MB/s]                              


In [4]:
# investigate the content of the downloaded dataset
artists
users

array(['00000c289a1829a808ac09c00daf10bc3c4e223b',
       '00001411dc427966b17297bf4d69e7e193135d89',
       '00004d2ac9316e22dc007ab2243d6fcb239e707d', ...,
       'ffff9af9ae04d263dae91cb838b1f3a6725f5ffb',
       'ffff9ef87a7d9494ada2f9ade4b9ff637c0759ac', 'sep 20, 2008'],
      shape=(358868,), dtype=object)

In [7]:
# return the dimensions of data
np.prod((artists.shape[0], users.shape[0]))

np.int64(104927620180)

In [15]:
# return the number of non-missing entries 
artist_user_plays.count_nonzero()

np.int64(17535605)

In [16]:
# investigate the proportion of non-zero entries
artist_user_plays.count_nonzero() / np.prod((artists.shape[0], users.shape[0]))

np.float64(0.0001671209636692248)

## Preparing the data
Okapi BM25 (Best Matching) scoring is a ranking algorithm used by search engines to estimate the relevance of items to a given search query, based on the frequency of occurrences and the size of the reference pool.  The origin of the algorithm is used in search terms in a pool of documents.

For completeness, the BM25 score of query $Q=\{q_1, \ldots, q_n\}$ for a document $D$ is calculated as:

$$\text{BM25}(D, Q) = \sum_{i=1}^{n} \frac{IDF(q_i) \cdot f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot (1 - b + b \cdot \frac{|D|}{\text{avgD}})},$$
where
- $IDF(q_i)$ is the inverse document frequency of term $q_i$.
- $f(q_i, D)$ is the term frequency of $q_i$ in the document $D$.
- $k_1$ and $b$ are parameters controlling term saturation and document length normalization.
- $D$ is the length of the document.
- $\text{avgD}$ is the average document length in the corpus.


In [18]:
from implicit.nearest_neighbours import bm25_weight
artist_user_plays = bm25_weight(artist_user_plays, K1=100, B=0.8)

user_plays = artist_user_plays.T.tocsr()
# using the weighting function for normalization

## Training the model with alternating least squares

In [22]:
from implicit.als import AlternatingLeastSquares

# using alternating least squares algorithm
model = AlternatingLeastSquares(factors=16,regularization=0.05,alpha=2.0)
model.fit(user_plays)

100%|██████████| 15/15 [00:20<00:00,  1.37s/it]


## Similar artists recommendation

In [ ]:
# generate similar artist recommendation
list(artists).index('beyonce')

45721

In [26]:
artist_id = 45721
ids, scores = model.similar_items(artist_id)

In [29]:
pd.DataFrame({'artist': artists[ids], 'scores': scores})

,artist,scores
0,beyonce,1.000000
1,ina,0.988127
2,ivena,0.980517
3,christina aguilerа,0.979854
4,jordin sparks ft chris brown,0.979444
5,brit & alex,0.975514
6,calvin13,0.974655
7,m. pokora,0.973028
8,musikk feat. john rock,0.972535
9,kevin rudolf feat. lil wayne,0.971463


## User-specific recommendation

In [30]:
# generate user-based recommendation
userid = 10
ids, scores = model.recommend(userid, user_plays[userid], N=10)

In [31]:
artists[ids]

array(['anna ternheim', 'moneybrother', 'hello saferide', 'johnossi',
       'laakso', 'marit bergman', 'broder daniel', 'familjen',
       'veronica maggio', 'billie the vision & the dancers'], dtype=object)